## 1. Install necessary libraries

In [1]:
# Install necessary libraries
!pip install --upgrade -q gspread scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 27.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.


## 2. Authenticate and access the Google Sheet

In [2]:
#Google Sheets Linking (Needs user authorization)
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [3]:
import pandas as pd

# Open the spreadsheet by URL
spreadsheet_url = 'https://docs.google.com/spreadsheets/d/1swxApDpgutQxNKr6GJHSr96PQZUxd_qcKAx8hlV7YOI/edit?usp=drive_link'
wb = gc.open_by_url(spreadsheet_url)
sheet = wb.get_worksheet(0)

# Load data into DataFrame
data = sheet.get_all_values()
df = pd.DataFrame(data[1:], columns=data[0])

print("Sanity Check - Original Data:")
display(df.head())

Sanity Check - Original Data:


,text,label_specific,label_generic
0,"Hello Michael, your account needs immediate at...",llm_phishing,phishing
1,"Dear Sarah, I trust this email finds you well....",llm_legit,legit
2,"Dear Ms. Jessica Davis, I trust this email fin...",llm_legit,legit
3,----------------------------------------------...,human_legit,legit
4,"Update account activity Hello jose@monkey.org,...",human_phishing,phishing


## 3. Vectorization (N-grams + TF-IDF)

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorization (N-grams + TF-IDF)
# Using unigrams and bigrams (1, 2)
print("Vectorizing text...")
tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=10000)
X = tfidf.fit_transform(df['text'].astype(str))

Vectorizing text...


## 4. Train (and Validate/Tune) Model 1: Binomial Logistic Model (Generic Labels)

While algorithms / 'Solvers' can auto find the best parameters... hyperparameters cannot be auto solved and must be trial-and-error found. This is our trial and error strategy:

*   **Stratified K-Fold**: This ensures that each fold of our cross-validation has the same proportion of observations with a given label as the whole dataset, which is crucial for handling potential class imbalances.
*   **GridSearchCV**: This allows us to systematically test different combinations of hyperparameters (for Binomial Logisitic Regression regularization strength `C`) to find the best configuration for our model.

This combined K-fold and GridSearch CV strategy performs parameter/model training-validation (fold) cycle k times for every combo of hyperparameters(gridsearch).

The scores of the k-folds per hyperparameter combo is averaged. Best averaged score means best hyperparameter combo.

In [5]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
import numpy as np

# This handles onyl 2 categories (e.g., 'phishing' vs 'legit')

# 1. Define the stratified k-fold cross-validation strategy
#    n_splits is k (10 folds), shuffle=True ensures randomness
skf_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 2. Define the hyperparameter grid for hyperparameter tuning / find best combo (specific hyperparameters for different models)
hyperparam_grid = {
    'C': [0.01, 0.1, 1, 10, 100, 1000],  # Inverse of regularization strength hyperparameter
}

# Encode labels numerically
y_generic_encoded = df['label_generic'].apply(lambda x: 1 if x == 'phishing' else 0)

# 3. Instantiate GridSearchCV with the stratified CV object
#    The 'cv' parameter takes the instantiated cross-validation generator
grid_search = GridSearchCV(
    estimator= LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42), #Define Model here (it auto trains)
    param_grid= hyperparam_grid,
    cv=skf_strategy, #input stratified cross validation strategy
    scoring={'f1': 'f1', 'roc_auc': 'roc_auc'}, #we will talk about scoring metrics more in our eval week.
    refit='roc_auc', # Specify which 1 metric to use for finding the best estimator
    verbose=1
)

# 4. Fit the model (do the grid search)
print("Training Binomial Model (Generic Labels) with Grid Search & Stratified K-Fold...")
grid_search.fit(X, y_generic_encoded)

# 5. View the best parameters found
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

# Use the best estimator as the final model
Binomial_Model_generic = grid_search.best_estimator_

Training Binomial Model (Generic Labels) with Grid Search & Stratified K-Fold...
Fitting 10 folds for each of 6 candidates, totalling 60 fits
Best parameters: {'C': 100}
Best CV accuracy: 0.9996


In [6]:
import pickle

# Save the Binomial_Model_generic model (into local folder - make sure to download then upload to drive)
with open('binomial_model_generic.pkl', 'wb') as f:
    pickle.dump(Binomial_Model_generic, f)

print("Binomial_Model_generic saved to binomial_model_specific.pkl")

Binomial_Model_generic saved to binomial_model_specific.pkl


## 5. Train (and Validate/Tune) Model 2: Multinomial Logistic Model (Specific Labels)

While algorithms / 'Solvers' can auto find the best parameters... hyperparameters cannot be auto solved and must be trial-and-error found. This is our trial and error strategy:

*   **Stratified K-Fold**: This ensures that each fold of our cross-validation has the same proportion of observations with a given label as the whole dataset, which is crucial for handling potential class imbalances.
*   **GridSearchCV**: This allows us to systematically test different combinations of hyperparameters (for Multinomial Logisitic Regression regularization strength `C`) to find the best configuration for our model.

This combined K-fold and GridSearch CV strategy performs parameter/model training-validation (fold) cycle k times for every combo of hyperparameters(gridsearch).

The scores of the k-folds per hyperparameter combo is averaged. Best averaged score means best hyperparameter combo.

In [7]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 1. Encode labels for multi-class (label_specific)
# This handles multiple categories (e.g., 'human_phishing', 'human_legit', 'llm_phishing', 'llm_legit'.)

le = LabelEncoder()
y_specific_encoded = le.fit_transform(df['label_specific'])

# 2. Define the stratified k-fold cross-validation strategy
skf_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 3. Define the hyperparameter grid for hyperparameter tuning / find best combo (specific hyperparameters for different models)
hyperparam_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
}

# 4. Instantiate GridSearchCV for Multinomial Logistic Regression
# We change the scoring to 'f1_macro' and 'roc_auc_ovr' for multi-class compatibility
grid_search_multi = GridSearchCV(
    estimator=LogisticRegression(solver='lbfgs', max_iter=2000, random_state=42), #model here with any truly set hyperparameter
    param_grid=hyperparam_grid,
    cv=skf_strategy,
    scoring={'f1_macro': 'f1_macro', 'roc_auc_ovr': 'roc_auc_ovr'},  #we will talk about scoring metrics more in our eval week.
    refit='roc_auc_ovr', # Specify which 1 metric to use for finding the best estimator
    verbose=1
)

# 5. Fit the model (do the grid search)
print("Training Multinomial Model (Specific Labels) with Grid Search & Stratified K-Fold...")
grid_search_multi.fit(X, y_specific_encoded)

# 6. View Results
print(f"Best parameters: {grid_search_multi.best_params_}")
print(f"Best CV F1-Macro: {grid_search_multi.best_score_:.4f}")

# Save the best estimator
Multinomial_Model_specific = grid_search_multi.best_estimator_

Training Multinomial Model (Specific Labels) with Grid Search & Stratified K-Fold...
Fitting 10 folds for each of 5 candidates, totalling 50 fits
Best parameters: {'C': 100}
Best CV F1-Macro: 0.9999


In [8]:
import pickle

# Save the Multinomial_Model_specific model (into local folder - make sure to download then upload to drive)
with open('multinomial_model_specific.pkl', 'wb') as f:
    pickle.dump(Multinomial_Model_specific, f)

print("Multinomial_Model_specific saved to multinomial_model_specific.pkl")

Multinomial_Model_specific saved to multinomial_model_specific.pkl


## Note 1. Dimensionality Reduction (TruncatedSVD) to be used after vectorization but before model training

We'll use TruncatedSVD to reduce the dimensionality of our TF-IDF vectors. This is particularly useful for sparse data like TF-IDF and can help with performance and mitigating the curse of dimensionality for algorithms like ***KNN, RF, and XGboost.***

In [9]:
from sklearn.decomposition import TruncatedSVD

# Perform TruncatedSVD for dimensionality reduction
# We'll reduce to 500 components as an example, this number can be tuned.
print("Performing TruncatedSVD...")
svd = TruncatedSVD(n_components=500, random_state=42)
X_reduced = svd.fit_transform(X)
print(f"Original dimensions: {X.shape}, Reduced dimensions: {X_reduced.shape}")

Performing TruncatedSVD...
Original dimensions: (3200, 10000), Reduced dimensions: (3200, 500)


## Note 2. Scaling (StandardScaler)

After dimensionality reduction, it's often beneficial to scale the features, especially for distance-based algorithms like ***KNN***. StandardScaler will transform the data to have a mean of 0 and a standard deviation of 1.

In [10]:
from sklearn.preprocessing import StandardScaler

# Perform StandardScaler scaling
print("Performing StandardScaler scaling...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reduced)
print(f"Scaled data shape: {X_scaled.shape}")

Performing StandardScaler scaling...
Scaled data shape: (3200, 500)


# 6. Evaluation #

Load in Eval/Test Set - (note for this example, the Train/Validate and Eval/Test Set are not Preprocessed [raw])



**6a.** Preprocess Test Data - Use same TFIDF as train set (vectorization not done yet for test/eval set)

**6b.** Define Custom Cost Function
Cost = (15 * False Negatives) + False Positives.

**6c.** Eval/Test Set Label Encoding (since not done yet)

**6d.** Load Pre-Trained Models

**6e.** Get Predictions and Probabilities for Labels on the Eval/Test Set

**6f.** Use Predictions and Probabilities for Labels on the Eval/Test Set to create Metrics of Model Fit / Goodness


In [ ]:
# Open the spreadsheet by URL
spreadsheet_url = 'https://docs.google.com/spreadsheets/d/1KsidfWCrywomIAbdlNg6Gn5rMER9E8gpI2AxGpNfjZ4/edit?gid=1833299105#gid=1833299105'
wb = gc.open_by_url(spreadsheet_url)
sheet = wb.get_worksheet(0)

# Load data into DataFrame
data = sheet.get_all_values()
eval_df = pd.DataFrame(data[1:], columns=data[0])

print("Sanity Check - Eval Set Data:")
display(eval_df.head())

### 6a. Preprocess Test Data - Use same TFIDF as train set (vecotrization not done yet for test/eval set)

#### TFIDF Again:
Transform the 'text' column of the `eval_df` DataFrame using the existing `tfidf` vectorizer used for the train validation set to create the test feature matrix `X_test`. If `tfidf` is not defined in the current kernel state, it will be re-initialized and fitted on the original training data.

This ensures that the trained tfidf vectorizer model used for training is applied/fitted to the test set.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Re-initialize and fit the TfidfVectorizer on the training data (df['text'])
# This ensures the vectorizer is properly fitted before transforming the test data.
print("Ensuring TF-IDF vectorizer is fitted on training data...")
tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=10000)
X = tfidf.fit_transform(df['text'].astype(str))
print("TF-IDF vectorizer fitted. Original training data (X) shape:", X.shape)

# Transform the 'text' column of the eval_df using the fitted tfidf vectorizer
print("Transforming evaluation data (eval_df['text']) using fitted TF-IDF vectorizer...")
X_test = tfidf.transform(eval_df['text'].astype(str))
print("X_test created with shape:", X_test.shape)


### 6b. Define Custom Cost Function

#### Function:
`Cost = (15 * False Negatives) + False Positives`.

A False Negative is defined as a Phishing email incorrectly marked as Legit. This is penalized 15x

A False Positive as a Legit email incorrectly marked as Phishing. This is penalized 1x


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

def calculate_custom_cost(y_true, y_pred):
    """
    Calculates a custom cost based on False Negatives and False Positives.

    Args:
        y_true (array-like): True labels (0 for 'legit', 1 for 'phishing').
        y_pred (array-like): Predicted labels (0 for 'legit', 1 for 'phishing').

    Returns:
        float: The calculated custom cost.
    """
    # Ensure y_true and y_pred are numpy arrays for element-wise comparison
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # False Negative: Actual Phishing (1), Predicted Legit (0)
    false_negatives = np.sum((y_true == 1) & (y_pred == 0))

    # False Positive: Actual Legit (0), Predicted Phishing (1)
    false_positives = np.sum((y_true == 0) & (y_pred == 1))

    # Calculate custom cost
    custom_cost = (15 * false_negatives) + false_positives

    return custom_cost

print("Function 'calculate_custom_cost' defined.")

### 6c. Eval/Test Set Label Encoding (since not done yet)

#### Encoding:
Eval set labels need to be turned into numerical expressions first (as done in training).

Prepare the true labels (`y_true_generic` and `y_true_specific`) from the `eval_df` for both generic and specific models.

For the generic model, 'phishing' should be mapped to 1 and 'legit' to 0.

For the specific model, the labels need to be encoded using the `LabelEncoder` that was previously fitted on the training data.



In [ ]:
from sklearn.preprocessing import LabelEncoder

# Prepare true labels for the generic model (0 for 'legit', 1 for 'phishing')
y_true_generic = eval_df['label_generic'].apply(lambda x: 1 if x == 'phishing' else 0)
print(f"y_true_generic shape: {y_true_generic.shape}")

# Prepare true labels for the specific model using the previously fitted LabelEncoder
# Assuming 'le' is still available from the training phase of the multinomial model.
# If 'le' is not available, it would need to be re-initialized and fitted on df['label_specific']

# Re-initialize and fit LabelEncoder for safety if it might not be in kernel state
# (though it should be if previous cells ran successfully)
le_eval = LabelEncoder()
le_eval.fit(df['label_specific']) # Fit on the original training labels to maintain consistency
y_true_specific = le_eval.transform(eval_df['label_specific'])
print(f"y_true_specific shape: {y_true_specific.shape}")

y_true_generic shape: (800,)
y_true_specific shape: (800,)


### 6d. Load Pre-Trained Models


In [ ]:
import pickle

# Load the Binomial_Model_generic
with open('binomial_model_generic.pkl', 'rb') as f:
    Binomial_Model_generic = pickle.load(f)
print("Binomial_Model_generic loaded successfully.")

# Load the Multinomial_Model_specific
with open('multinomial_model_specific.pkl', 'rb') as f:
    Multinomial_Model_specific = pickle.load(f)
print("Multinomial_Model_specific loaded successfully.")

Binomial_Model_generic loaded successfully.
Multinomial_Model_specific loaded successfully.


### 6e. Get Predictions and Probabilities for Labels on the Eval/Test Set

#### Explanation:
We now use the trained models to predict the labels of the test set.
Scikit-learn models enable use to extract both the `Predictions` and the `Probabilities` for the label on a test instance/row/email.

We will then use these and create evaluation metrics from them.

In [ ]:
import numpy as np

# Make predictions with Binomial_Model_generic
print("Making predictions with Binomial_Model_generic...")
y_pred_generic = Binomial_Model_generic.predict(X_test)
y_proba_generic = Binomial_Model_generic.predict_proba(X_test)[:, 1] # Get probabilities for the 'phishing' class (1)
print(f"Predictions for generic model generated. Shape: {y_pred_generic.shape}")

# Make predictions with Multinomial_Model_specific
print("Making predictions with Multinomial_Model_specific...")
y_pred_specific = Multinomial_Model_specific.predict(X_test)
y_proba_specific = Multinomial_Model_specific.predict_proba(X_test) # Get probabilities for all classes
print(f"Predictions for specific model generated. Shape: {y_pred_specific.shape}")

Making predictions with Binomial_Model_generic...
Predictions for generic model generated. Shape: (800,)
Making predictions with Multinomial_Model_specific...
Predictions for specific model generated. Shape: (800,)


### 6f. Use Predictions and Probabilities for Labels on the Eval/Test Set to create Metrics of Model Fit / Goodness

#### Metrics:

Binomial metrics: F1-score, ROC-AUC, custom cost.

Multinomial (more than 2 label) metrics: F1-macro score, ROC-AUC score (using OvR strategy for multi-class), and the custom cost.


In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

# Evaluate Binomial_Model_generic
print("\n--- Binomial Model (Generic Labels) Evaluation ---")
f1_generic = f1_score(y_true_generic, y_pred_generic)
roc_auc_generic = roc_auc_score(y_true_generic, y_proba_generic)
cost_generic = calculate_custom_cost(y_true_generic, y_pred_generic)

print(f"F1 Score (Generic): {f1_generic:.4f}")
print(f"ROC-AUC (Generic): {roc_auc_generic:.4f}")
print(f"Custom Cost (Generic): {cost_generic}")


--- Binomial Model (Generic Labels) Evaluation ---
F1 Score (Generic): 0.9975
ROC-AUC (Generic): 0.9998
Custom Cost (Generic): 30


In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

# Evaluate Multinomial_Model_specific
print("\n--- Multinomial Model (Specific Labels) Evaluation ---")

# F1 Score (macro average for multi-class)
f1_specific = f1_score(y_true_specific, y_pred_specific, average='macro')

# ROC-AUC (OvR for multi-class)
# y_proba_specific needs to be the probabilities for each class
roc_auc_specific = roc_auc_score(y_true_specific, y_proba_specific, multi_class='ovr', average='macro')

# Custom Cost: Convert specific labels to generic for cost calculation
# Get the class names from the LabelEncoder
specific_classes = le_eval.classes_

# Create a mapping from encoded specific labels to binary generic labels
# 'phishing' (1) for human_phishing, llm_phishing
# 'legit' (0) for human_legit, llm_legit

specific_to_generic_map = {}
for i, class_name in enumerate(specific_classes):
    if 'phishing' in class_name:
        specific_to_generic_map[i] = 1 # Phishing
    else:
        specific_to_generic_map[i] = 0 # Legit

# Apply the mapping to true and predicted specific labels
y_true_specific_binary = np.array([specific_to_generic_map[label] for label in y_true_specific])
y_pred_specific_binary = np.array([specific_to_generic_map[label] for label in y_pred_specific])

# Calculate custom cost using the binary mapped labels
cost_specific = calculate_custom_cost(y_true_specific_binary, y_pred_specific_binary)

print(f"F1 Score (Specific, Macro): {f1_specific:.4f}")
print(f"ROC-AUC (Specific, OvR, Macro): {roc_auc_specific:.4f}")
print(f"Custom Cost (Specific): {cost_specific}")


--- Multinomial Model (Specific Labels) Evaluation ---
F1 Score (Specific, Macro): 0.9962
ROC-AUC (Specific, OvR, Macro): 1.0000
Custom Cost (Specific): 30
